# Chapter 2 Practical 05: SBERT Semantic Recommender

Learning objectives:
- Explain lexical similarity versus semantic similarity.
- Encode movie descriptions with SBERT when available.
- Fall back to TF-IDF when `sentence-transformers` or the model is unavailable.
- Compare TF-IDF and semantic recommendation results.

Slide connection: deep content models, SBERT/BERT embeddings, semantic similarity, and practical fallback design.


TF-IDF works with shared words. SBERT can also capture related meanings, such as `astronaut`, `space`, `orbit`, and `Mars`.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("../data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("chapter_02_content_based/data")

movies = pd.read_csv(DATA_DIR / "movies_chapter2.csv")
movies.head()


First build a TF-IDF baseline that always works in a basic Python environment.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies["semantic_text"] = movies["title"] + ". " + movies["description"] + " Keywords: " + movies["keywords"]

tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies["semantic_text"])
tfidf_similarity = cosine_similarity(tfidf_matrix)


Now try SBERT. If the package is missing or the model cannot be downloaded, the notebook continues with the TF-IDF fallback.


In [ ]:
embedding_source = "tfidf fallback"
sbert_similarity = tfidf_similarity

try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = model.encode(movies["semantic_text"].tolist(), show_progress_bar=False)
    sbert_similarity = cosine_similarity(embeddings)
    embedding_source = "sentence-transformers/all-MiniLM-L6-v2"
except Exception as exc:
    print("SBERT is not available in this environment.")
    print("Using TF-IDF fallback instead.")
    print(type(exc).__name__, str(exc)[:160])

embedding_source


Use the same recommendation function for either similarity matrix.


In [ ]:
def recommend(title, similarity_matrix, n=5):
    idx = movies.index[movies["title"].eq(title)][0]
    scores = sorted(enumerate(similarity_matrix[idx]), key=lambda x: x[1], reverse=True)
    return pd.DataFrame([
        {"query_movie": title, "recommended_movie": movies.loc[i, "title"], "score": round(float(score), 3)}
        for i, score in scores[1:n+1]
    ])

recommend("Interstellar", sbert_similarity)


Compare lexical TF-IDF results with semantic SBERT results. If SBERT is not available, both columns will show the fallback behavior.


In [ ]:
tfidf_results = recommend("Interstellar", tfidf_similarity, n=5).rename(columns={
    "recommended_movie": "tfidf_recommendation",
    "score": "tfidf_score",
})
sbert_results = recommend("Interstellar", sbert_similarity, n=5).rename(columns={
    "recommended_movie": "semantic_recommendation",
    "score": "semantic_score",
})

pd.concat([
    tfidf_results[["tfidf_recommendation", "tfidf_score"]],
    sbert_results[["semantic_recommendation", "semantic_score"]],
], axis=1)


Zero-shot style semantic search uses a text query instead of an input movie.


In [ ]:
queries = ["movies about space exploration", "romantic drama about lifelong love"]

if embedding_source.startswith("sentence-transformers"):
    query_embeddings = model.encode(queries, show_progress_bar=False)
    query_scores = cosine_similarity(query_embeddings, embeddings)
else:
    query_matrix = tfidf.transform(queries)
    query_scores = cosine_similarity(query_matrix, tfidf_matrix)

rows = []
for q_idx, query in enumerate(queries):
    best = query_scores[q_idx].argsort()[::-1][:4]
    for movie_idx in best:
        rows.append({"query": query, "movie": movies.loc[movie_idx, "title"], "score": round(float(query_scores[q_idx, movie_idx]), 3)})

pd.DataFrame(rows)


## What did we learn?

- TF-IDF is lexical: shared words drive similarity.
- SBERT is semantic: related meanings can be close even with different words.
- Optional models should have fallback logic so teaching notebooks still run.

Exercises:
1. Try the query `mind-bending action movie`.
2. Add a new movie description that avoids the word `space` but is clearly about astronauts.
